# ImageNette Data Overview

This notebook is intentionally simple.

It does three things:
- gets ImageNette in the easiest reasonable way for this TensorFlow-based repo
- shows the dataset structure and some example images
- prints enough statistics that we understand what we are working with

Conventions:
- the notebook is committed to Git
- cached data lives under `data/raw/` and stays out of Git because `data/` is already ignored
- outputs should be cleared before committing


## 1. Setup and simplest dataset download

Instead of manually downloading and extracting the archive, we let Keras handle that for us.


In [ ]:
import random
import statistics
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "bayesian_cv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


def print_tree(root: Path, depth: int = 2, max_entries: int = 10, prefix: str = "") -> None:
    print(f"{prefix}{root.name}/")
    if depth == 0:
        return

    entries = sorted(root.iterdir(), key=lambda path: (not path.is_dir(), path.name.lower()))
    for entry in entries[:max_entries]:
        if entry.is_dir():
            print_tree(entry, depth=depth - 1, max_entries=max_entries, prefix=prefix + "  ")
        else:
            print(f"{prefix}  {entry.name}")

    if len(entries) > max_entries:
        print(f"{prefix}  ... ({len(entries) - max_entries} more entries)")


def list_image_files(folder: Path) -> list[Path]:
    return sorted(
        path
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in {".jpeg", ".jpg", ".png", ".bmp"}
    )


random.seed(255)

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
IMAGENETTE_MD5 = "e793b78cc4c9e9a4ccc0c1155377a412"
IMAGENETTE_LABELS = {
    "n01440764": "tench",
    "n02102040": "English springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}

RUNNING_IN_COLAB = "google.colab" in sys.modules
REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

download_path = Path(
    tf.keras.utils.get_file(
        origin=IMAGENETTE_URL,
        file_hash=IMAGENETTE_MD5,
        cache_dir=str(RAW_DATA_DIR),
        cache_subdir="",
        extract=True,
    )
)

if download_path.is_dir():
    IMAGENETTE_DIR = download_path / "imagenette2-160"
else:
    IMAGENETTE_DIR = download_path.with_suffix("")

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"Repository root: {REPO_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"ImageNette directory: {IMAGENETTE_DIR}")


## 2. Raw folder structure


In [ ]:
print_tree(IMAGENETTE_DIR, depth=2, max_entries=12)


## 3. Splits, classes, and image counts


In [ ]:
split_dirs = sorted(path for path in IMAGENETTE_DIR.iterdir() if path.is_dir())
dataset_summary = {}
all_class_names = set()

for split_dir in split_dirs:
    class_counts = {}
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        count = len(list_image_files(class_dir))
        class_counts[class_dir.name] = count
        all_class_names.add(class_dir.name)
    dataset_summary[split_dir.name] = class_counts

print(f"Discovered splits: {[path.name for path in split_dirs]}")
print(f"Discovered class folders: {len(all_class_names)}")

print("\nClass folder mapping:")
for class_name in sorted(all_class_names):
    print(f"- {class_name}: {IMAGENETTE_LABELS.get(class_name, class_name)}")

print("\nPer-split counts:")
for split_name, class_counts in dataset_summary.items():
    print(f"\n[{split_name}] total images: {sum(class_counts.values())}")
    for class_name, count in class_counts.items():
        label = IMAGENETTE_LABELS.get(class_name, class_name)
        print(f"  - {class_name} ({label}): {count}")


## 4. Example images

We show one random image per class from the `train` split.


In [ ]:
preferred_split = "train" if "train" in dataset_summary else next(iter(dataset_summary))
sample_paths = []

for class_name in sorted(dataset_summary[preferred_split]):
    class_dir = IMAGENETTE_DIR / preferred_split / class_name
    class_images = list_image_files(class_dir)
    sample_paths.append(random.choice(class_images))

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for ax, image_path in zip(axes, sample_paths):
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image)
    class_name = image_path.parent.name
    label = IMAGENETTE_LABELS.get(class_name, class_name)
    ax.set_title(f"{label}\n{class_name}", fontsize=10)
    ax.axis("off")

plt.suptitle(f"One sample per class from the '{preferred_split}' split")
plt.tight_layout()
plt.show()


## 5. Image sizes and aspect ratios

This is a simple pass through the dataset to understand image dimensions.


In [ ]:
image_records = []

for split_dir in split_dirs:
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        for image_path in list_image_files(class_dir):
            image = Image.open(image_path)
            width, height = image.size
            image_records.append(
                {
                    "split": split_dir.name,
                    "class_name": class_dir.name,
                    "path": image_path,
                    "width": width,
                    "height": height,
                    "aspect_ratio": width / height,
                }
            )

print(f"Total readable images scanned: {len(image_records)}")

for split_name in dataset_summary:
    records = [record for record in image_records if record["split"] == split_name]
    widths = [record["width"] for record in records]
    heights = [record["height"] for record in records]
    aspect_ratios = [record["aspect_ratio"] for record in records]

    print(f"\n[{split_name}] size summary")
    print(f"- images: {len(records)}")
    print(f"- width min/max/mean/median: {min(widths)} / {max(widths)} / {statistics.fmean(widths):.2f} / {statistics.median(widths):.2f}")
    print(f"- height min/max/mean/median: {min(heights)} / {max(heights)} / {statistics.fmean(heights):.2f} / {statistics.median(heights):.2f}")
    print(f"- aspect ratio min/max/mean/median: {min(aspect_ratios):.3f} / {max(aspect_ratios):.3f} / {statistics.fmean(aspect_ratios):.3f} / {statistics.median(aspect_ratios):.3f}")

most_common_sizes = Counter((record["width"], record["height"]) for record in image_records)
print("\nMost common image sizes:")
for (width, height), count in most_common_sizes.most_common(10):
    print(f"- {width}x{height}: {count} images")


## 6. Project notes for later steps

This notebook does not build the training pipeline. It only helps us understand the data before that step.


In [ ]:
from bayesian_cv.config import get_config

config = get_config()

print("Project alignment summary")
print("- ImageNette is a 10-class dataset.")
print(f"- This notebook found {len(all_class_names)} class folders.")
print(f"- Current project config uses num_classes={config.num_classes} in bayesian_cv/config.py.")
print("- Raw ImageNette uses train/ and val/ folders.")
print("- The current repo skeleton expects data/train, data/val, and data/test.")
print("- Recommended raw-data convention for now: data/raw/imagenette2-160/...")
print("- Downstream target convention later:")
print("  data/train/<class>/...")
print("  data/val/<class>/...")
print("  data/test/<class>/...")


## 7. How to use this data next

Now that we know the dataset structure, the next step is to make the training code read it in a consistent way.

What we know now:
- ImageNette has 10 classes, so `num_classes=10` is the correct project setting when using the full dataset
- the raw dataset currently lives under `data/raw/...`
- the raw structure contains `train/` and `val/` folders

What we can do next in the project:
- either load the raw ImageNette folders directly from `data/raw/...`
- or copy/split them into the repo convention `data/train`, `data/val`, and `data/test`

For this project, the simplest next step is usually:
1. use `data/raw/.../train` as the training split
2. use `data/raw/.../val` as the validation split
3. later decide whether to create a separate test split from the training data or keep `val` for final evaluation in a smaller prototype

In other words, this notebook gives us the information needed to build `bayesian_cv/data.py` next.
